Ideia pro script da cabos usando as estacas:
1. Recolocar os trackers tagueando com TRK
2. Colocar estacas e tagueando elas também

In [25]:
from DWG_to_Excel_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - Eletromecânico - DC.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - Eletromecânico - DC.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - Eletromecânico - DC.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


In [26]:
ELM_df = pd.DataFrame(data)

print(ELM_df.describe())
print(ELM_df.info())
print(ELM_df['name'].value_counts())

                  X              Y      rotation       start_x        start_y  \
count  12860.000000   12860.000000  12860.000000  39667.000000   39667.000000   
mean   35060.910221  128080.129624     73.595645  35060.896783  128096.756071   
std      593.210525     951.628712     88.566948    597.164208     959.100003   
min    33667.550000  125996.966474      0.000000  33667.550000  125996.966474   
25%    34622.695128  127293.973600      0.000000  34614.294008  127314.491400   
50%    34998.590000  128003.276275      0.000000  34998.590000  128067.624300   
75%    35483.686031  128960.985000    180.000000  35489.986150  128982.969200   
max    36625.890000  129787.710488    270.000000  36625.890000  129808.228288   

              end_x          end_y        length  
count  39667.000000   39667.000000  39667.000000  
mean   35060.811407  128106.715200     57.665334  
std      596.608151     957.878980     55.095058  
min    33669.246800  125996.966474      0.000000  
25%    34620.05

In [27]:
string_mask = (ELM_df['name'] == "1x64p") | (ELM_df['name'] == "EMF_tracker_32p")

strings_df = ELM_df[string_mask].copy()

strings_df.rename(columns={'X': 'Position_X', 'Y': 'Position_Y'}, inplace=True)
strings_df.drop(columns=['rotation', 'type', 'handle', 'layer', 'QBT', 'DSJ', 'PVM'], inplace=True)
#strings_df.to_excel(r"C:\Users\Usuario\Desktop\strings.xlsx")

strings_df['UFV'] = strings_df['UFV'].astype(int)
strings_df['PST'] = strings_df['PST'].astype(int)
strings_df['INV'] = strings_df['INV'].astype(int)
strings_df['Line'] = round(strings_df['Position_X'],0)
print(strings_df.describe())
print(strings_df.info())
print(strings_df)


         Position_X     Position_Y      UFV           PST           INV  \
count  12367.000000   12367.000000  12367.0  12367.000000  12367.000000   
mean   35060.863710  128079.507805      0.0     15.012857      8.497291   
std      593.305674     951.731315      0.0      8.364836      4.608019   
min    33667.550000  126018.800000      0.0      1.000000      1.000000   
25%    34622.695128  127293.973600      0.0      8.000000      4.000000   
50%    35003.257868  128003.276275      0.0     15.000000      9.000000   
75%    35483.686150  128960.985000      0.0     22.000000     12.000000   
max    36625.890000  129787.710488      0.0     29.000000     16.000000   

       start_x  start_y  end_x  end_y  length          Line  
count      0.0      0.0    0.0    0.0     0.0  12367.000000  
mean       NaN      NaN    NaN    NaN     NaN  35060.836743  
std        NaN      NaN    NaN    NaN     NaN    593.302794  
min        NaN      NaN    NaN    NaN     NaN  33668.000000  
25%        NaN

In [28]:
inverter_mask = (ELM_df['name'] == "EMF397_inversor")

inverter_df = ELM_df[inverter_mask].copy()
inverter_df.rename(columns={'X': 'inverter_x', 'Y': 'inverter_y'}, inplace=True)
inverter_df.drop(columns=['name', 'rotation', 'type', 'handle', 'layer'], inplace=True)
inverter_df["GRP"] = inverter_df["UFV"].astype(str) + "." + inverter_df["PST"].astype(str).str.zfill(2) + "." + inverter_df["INV"].astype(str).str.zfill(2)
inverter_df['inverter_line'] = round(inverter_df['inverter_x'],0)
inverter_df.sort_values(by=['inverter_y', 'inverter_x'], ascending=[False, True], inplace=True)
print(inverter_df.describe())
print(inverter_df.info())
print(inverter_df)

         inverter_x     inverter_y  start_x  start_y  end_x  end_y  length  \
count    464.000000     464.000000      0.0      0.0    0.0    0.0     0.0   
mean   35060.758106  128096.236730      NaN      NaN    NaN    NaN     NaN   
std      593.480608     951.683275      NaN      NaN    NaN    NaN     NaN   
min    33680.153807  125996.966474      NaN      NaN    NaN    NaN     NaN   
25%    34620.590000  127272.127575      NaN      NaN    NaN    NaN     NaN   
50%    34994.623934  128067.624300      NaN      NaN    NaN    NaN     NaN   
75%    35477.386994  128995.651570      NaN      NaN    NaN    NaN     NaN   
max    36569.191051  129708.743463      NaN      NaN    NaN    NaN     NaN   

       inverter_line  
count     464.000000  
mean    35060.741379  
std       593.456531  
min     33680.000000  
25%     34621.000000  
50%     34994.500000  
75%     35477.000000  
max     36569.000000  
<class 'pandas.DataFrame'>
Index: 464 entries, 7111 to 6715
Data columns (total 15 columns

In [29]:
strings_df["GRP"] = strings_df["UFV"].astype(str) + "." + strings_df["PST"].astype(str).str.zfill(2) + "." + strings_df["INV"].astype(str).str.zfill(2)
strings_df.sort_values(by=['Line', 'Position_Y'], ascending=[True, False], inplace=True)


#Different Lines
condition1 = strings_df["Line"] != strings_df["Line"].shift()

#Different PSTs
condition2 = strings_df["GRP"] != strings_df["GRP"].shift()

#Break in the Structure
condition3 = abs(strings_df["Position_Y"] - strings_df["Position_Y"].shift()) > 44

break_condition_1 = condition1 | condition2 | condition3
break_condition_2 = condition1 | condition3

strings_df['SubGroup'] = break_condition_1.cumsum() # The SubGroup represents strings that are on the same Line, are from the same inverter and don't have a break between them. Used to count the number of strings that are going down the pile in the end.
strings_df['Group'] = break_condition_2.cumsum() # The Group represents strings that are on the same Line and don't have a break between them. Used to calculate where the start and end piles are.
print(strings_df)
strings_df.to_excel(r"C:\Users\Usuario\Desktop\teste.xlsx")

                  name   Position_X     Position_Y  UFV  PST  INV      GRP  \
9441   EMF_tracker_32p  33667.55000  129034.800000    0    7    1  0.07.01   
9442   EMF_tracker_32p  33667.55000  128973.818545    0    7    1  0.07.01   
8156             1x64p  33667.55349  128930.306089    0    7    1  0.07.01   
8155             1x64p  33667.55349  128887.804089    0    7    1  0.07.01   
8212             1x64p  33667.55349  128844.275039    0    7    1  0.07.01   
...                ...          ...            ...  ...  ...  ...      ...   
10403  EMF_tracker_32p  36600.69000  129536.320000    0    3    3  0.03.03   
10404  EMF_tracker_32p  36606.99000  129536.320000    0    3    3  0.03.03   
10405  EMF_tracker_32p  36613.29000  129536.320000    0    3    3  0.03.03   
10406  EMF_tracker_32p  36619.59000  129536.320000    0    3    3  0.03.03   
10407  EMF_tracker_32p  36625.89000  129536.320000    0    3    3  0.03.03   

       start_x  start_y  end_x  end_y  length     Line  SubGrou

In [30]:
distance_32p = 20.5048 # Distance between the middle point of the string and the 32p tracker's pile.
distance_64p = 20.5178 # Distance between the middle point of the string and the 64p tracker's pile.

strings_df['N exit'] = np.where(
    strings_df['name'] == '1x32p',
    strings_df['Position_Y'] + distance_32p,
    strings_df['Position_Y'] + distance_64p
)

strings_df['S exit'] = np.where(
    strings_df['name'] == '1x32p',
    strings_df['Position_Y'] - distance_32p,
    strings_df['Position_Y'] - distance_64p
)

strings_df['SubGroup - Position Y - N'] = strings_df.groupby("SubGroup")["N exit"].transform('max')
strings_df['SubGroup - Position Y - S'] = strings_df.groupby("SubGroup")["S exit"].transform('min')

strings_df['Group - Position Y - N'] = strings_df.groupby("Group")["N exit"].transform('max')
strings_df['Group - Position Y - S'] = strings_df.groupby("Group")["S exit"].transform('min')

cables_df = strings_df.merge(inverter_df, on="GRP")

strings_df['No. of Strings'] = strings_df.groupby("SubGroup")['Line'].transform('count')

print(strings_df)

                  name   Position_X     Position_Y  UFV  PST  INV      GRP  \
9441   EMF_tracker_32p  33667.55000  129034.800000    0    7    1  0.07.01   
9442   EMF_tracker_32p  33667.55000  128973.818545    0    7    1  0.07.01   
8156             1x64p  33667.55349  128930.306089    0    7    1  0.07.01   
8155             1x64p  33667.55349  128887.804089    0    7    1  0.07.01   
8212             1x64p  33667.55349  128844.275039    0    7    1  0.07.01   
...                ...          ...            ...  ...  ...  ...      ...   
10403  EMF_tracker_32p  36600.69000  129536.320000    0    3    3  0.03.03   
10404  EMF_tracker_32p  36606.99000  129536.320000    0    3    3  0.03.03   
10405  EMF_tracker_32p  36613.29000  129536.320000    0    3    3  0.03.03   
10406  EMF_tracker_32p  36619.59000  129536.320000    0    3    3  0.03.03   
10407  EMF_tracker_32p  36625.89000  129536.320000    0    3    3  0.03.03   

       start_x  start_y  end_x  ...     Line  SubGroup  Group  

In [31]:
groups_df = strings_df.groupby("SubGroup")[['GRP','Position_X', 'Position_Y', 'SubGroup - Position Y - N', 'SubGroup - Position Y - S', 'Group - Position Y - N', 'Group - Position Y - S', 'No. of Strings', 'Line']].first()
groups_df.rename(columns={'Position_X': 'X', 'Position_Y': 'Y', 'Group - Position Y - N': 'Y-N-TRK', 'Group - Position Y - S': 'Y-S-TRK', 'SubGroup - Position Y - N': 'Y-N-STR', 'SubGroup - Position Y - S': 'Y-S-STR'}, inplace=True)
print(f'Maximum number of strings in a group: {groups_df['No. of Strings'].max()}\n')
print(groups_df)

Maximum number of strings in a group: 8

              GRP         X              Y        Y-N-STR        Y-S-STR  \
SubGroup                                                                   
1         0.07.01  33667.55  129034.800000  129055.317800  129014.282200   
2         0.07.01  33667.55  128973.818545  128994.336345  128781.255239   
3         0.07.01  33673.85  129034.800000  129055.317800  129014.282200   
4         0.07.01  33673.85  128973.818545  128994.336345  128781.255239   
5         0.07.01  33680.15  129034.800000  129055.317800  129014.282200   
...           ...       ...            ...            ...            ...   
4715      0.03.03  36600.69  129536.320000  129556.837800  129515.802200   
4716      0.03.03  36606.99  129536.320000  129556.837800  129515.802200   
4717      0.03.03  36613.29  129536.320000  129556.837800  129515.802200   
4718      0.03.03  36619.59  129536.320000  129556.837800  129515.802200   
4719      0.03.03  36625.89  129536.320000  129

In [32]:
duct_map = {
    1: '-1-',
    2: '-2-',
    3: '-3-',
    4: '-3-1-',
    5: '-3-2-',
    6: '-3-3-',
    7: '-3-3-1-',
    8: '-3-3-2-',
    9: '-3-3-3-',
    10: '-3-3-3-1-',
    11: '-3-3-3-2-',
    12: '-3-3-3-3-',
    13: '-3-3-3-3-1-',
    14: '-3-3-3-3-2-',
    15: '-3-3-3-3-3-',
    16: '-3-3-3-3-3-1-',
    17: '-3-3-3-3-3-2-',
    18: '-3-3-3-3-3-3-',
    19: '-3-3-3-3-3-3-1-',
    20: '-3-3-3-3-3-3-2-',
}

groups_df.loc[:,'Duct Definition'] = groups_df['No. of Strings'].map(duct_map)
groups_df['Number of Ducts'] = groups_df['Duct Definition'].astype(str).str.count('-')-1
groups_df = groups_df.merge(inverter_df, on='GRP')
groups_df['N or S or M'] = np.where(
    abs(groups_df['Y-N-TRK'] - groups_df['inverter_y']) > abs(groups_df['Y-S-TRK'] - groups_df['inverter_y']),
    "S",
    np.where(
        (groups_df['inverter_y'] < groups_df['Y-N-TRK']) & (groups_df['inverter_y'] > groups_df['Y-S-TRK']),
        "M",
        "N"
    )
)

# THIS FINDS THE CLOSEST Y TO EXIT THE STRING.
# CALCULATING WHICH ONE IS THE CLOSEST ONE (Y-N-TRK, Y-S-TRK OR Y).
groups_df['closest_y'] = np.where(
    abs(groups_df['Y-N-TRK'] - groups_df['inverter_y']) < abs(groups_df['Y'] - groups_df['inverter_y']),
    groups_df['Y-N-TRK'],
    np.where(
        abs(groups_df['Y'] - groups_df['inverter_y']) < abs(groups_df['Y-S-TRK'] - groups_df['inverter_y']),
        groups_df['Y'],
        groups_df['Y-S-TRK']
    )
)

groups_df.sort_values(['GRP', 'Line'], inplace = True)

print(groups_df.head(20))


          GRP            X              Y        Y-N-STR        Y-S-STR  \
4346  0.01.01  36016.90525  129773.091488  129793.609288  129710.071688   
4351  0.01.01  36023.20525  129773.091488  129793.609288  129710.071688   
4356  0.01.01  36029.50525  129773.091488  129793.609288  129710.071688   
4361  0.01.01  36035.80525  129773.091488  129793.609288  129710.071688   
4367  0.01.01  36042.10525  129773.091488  129793.609288  129710.071688   
4373  0.01.01  36048.40525  129773.091488  129793.609288  129710.071688   
4379  0.01.01  36054.70525  129773.091488  129793.609288  129710.071688   
4386  0.01.01  36061.00525  129773.091488  129793.609288  129710.071688   
4392  0.01.01  36067.30525  129773.091488  129793.609288  129710.071688   
4398  0.01.01  36073.60525  129773.091488  129793.609288  129710.071688   
4405  0.01.01  36079.90525  129773.091488  129793.609288  129710.071688   
4412  0.01.01  36086.20525  129773.091488  129793.609288  129710.071688   
4419  0.01.01  36092.5052

Breaking the trenches into segments to indicate how many strings there are in each one.
=

In [33]:
"""
Adds cumulative sum column where values sum towards the inverter_line, calculated separately for each distinct group/segment.
"""
groups_df['STR_CUM_TO_INV'] = 0

# Group by GRP
# Each GRP represents a distinct physical segment with its own inverter
for grp_name in groups_df['GRP'].unique():
    # Get rows belonging to this GRP
    grp_mask = groups_df['GRP'] == grp_name
    grp_indices = groups_df[grp_mask].index

    if len(grp_indices) == 0:
        continue

    # Get the inverter_line for this GRP (should be consistent within group)
    # Using the first occurrence's inverter_line as the target for this group
    inv_line = groups_df.loc[grp_indices[0], 'inverter_line']

    # Separate rows by position relative to inverter_line
    before_inv = []
    at_inv = []
    after_inv = []

    for idx in grp_indices:
        x_val = groups_df.loc[idx, 'Line']
        if x_val < inv_line:
            before_inv.append(idx)
        elif x_val > inv_line:
            after_inv.append(idx)
        else:  # x_val == inv_line
            at_inv.append(idx)

    # Sort before_inv by X ascending (farthest to closest to inverter)
    before_inv.sort(key=lambda idx: groups_df.loc[idx, 'Line'])

    # Sort after_inv by X descending (farthest to closest to inverter)
    after_inv.sort(key=lambda idx: groups_df.loc[idx, 'Line'], reverse=True)

    # Calculate cumulative sums for before inverter (left to right)
    cumsum = 0
    for idx in before_inv:
        cumsum += groups_df.loc[idx, 'No. of Strings']
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = cumsum

    # At inverter line: just the value itself
    for idx in at_inv:
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = groups_df.loc[idx, 'No. of Strings']

    # Calculate cumulative sums for after inverter (right to left)
    cumsum = 0
    for idx in after_inv:
        cumsum += groups_df.loc[idx, 'No. of Strings']
        groups_df.loc[idx, 'STR_CUM_TO_INV'] = cumsum


# Verify the results
print(groups_df[['GRP', 'X', 'No. of Strings', 'inverter_line', 'STR_CUM_TO_INV']].to_string())
max_number_of_strings = groups_df['STR_CUM_TO_INV'].max()
print(groups_df['STR_CUM_TO_INV'].value_counts())
groups_df.to_excel(r"C:\Users\Usuario\Desktop\teste1.xlsx")

          GRP             X  No. of Strings  inverter_line  STR_CUM_TO_INV
4346  0.01.01  36016.905250               2        36055.0               2
4351  0.01.01  36023.205250               2        36055.0               4
4356  0.01.01  36029.505250               2        36055.0               6
4361  0.01.01  36035.805250               2        36055.0               8
4367  0.01.01  36042.105250               2        36055.0              10
4373  0.01.01  36048.405250               2        36055.0              12
4379  0.01.01  36054.705250               2        36055.0               2
4386  0.01.01  36061.005250               2        36055.0              12
4392  0.01.01  36067.305250               2        36055.0              10
4398  0.01.01  36073.605250               2        36055.0               8
4405  0.01.01  36079.905250               2        36055.0               6
4412  0.01.01  36086.205250               2        36055.0               4
4419  0.01.01  36092.5052

In [34]:
distance2 = 0.8805 # Distance between the first/last pile and the first/last module

cables_df.drop(columns=['UFV_x', 'UFV_y', 'PST_x', 'PST_y', 'INV_x', 'INV_y', 'SubGroup', 'Group'], inplace=True)
cables_df.rename(columns={'Position_X': 'X', 'Position_Y': 'Y', 'Group - Position Y - N': 'Y-N-TRK', 'Group - Position Y - S': 'Y-S-TRK', 'SubGroup - Position Y - N': 'Y-N-STR', 'SubGroup - Position Y - S': 'Y-S-STR'}, inplace=True)
cables_df['N or S or M'] = np.where(
    abs(cables_df['N exit'] - cables_df['inverter_y']) > abs(cables_df['S exit'] - cables_df['inverter_y']),
    "S",
    np.where(
        (cables_df['inverter_y'] < cables_df['N exit']) & (cables_df['inverter_y'] > cables_df['S exit']),
        "M",
        "N"
    )
)

# THIS FINDS THE CLOSEST Y TO EXIT THE STRING.
# CALCULATING WHICH ONE IS THE CLOSEST ONE (Y-N-TRK, Y-S-TRK OR Y).
cables_df['closest_y'] = np.where(
    abs(cables_df['Y-N-TRK'] - cables_df['inverter_y']) < abs(cables_df['Y'] - cables_df['inverter_y']),
    cables_df['Y-N-TRK'],
    np.where(
        abs(cables_df['Y'] - cables_df['inverter_y']) < abs(cables_df['Y-S-TRK'] - cables_df['inverter_y']),
        cables_df['Y'],
        cables_df['Y-S-TRK']
    )
)


cables_df['Y-N-STR'] = cables_df['Y-N-STR'] + distance2
cables_df['Y-S-STR'] = cables_df['Y-S-STR'] - distance2

cables_df.sort_values(['GRP', 'Line'], inplace = True)
print(cables_df.head())

        name            X              Y      GRP  start_x_x  start_y_x  \
11421  1x64p  36016.90525  129773.091488  0.01.01        NaN        NaN   
11422  1x64p  36016.90525  129730.589488  0.01.01        NaN        NaN   
11433  1x64p  36023.20525  129773.091488  0.01.01        NaN        NaN   
11434  1x64p  36023.20525  129730.589488  0.01.01        NaN        NaN   
11446  1x64p  36029.50525  129773.091488  0.01.01        NaN        NaN   

       end_x_x  end_y_x  length_x     Line  ...  DSJ  PVM  start_x_y  \
11421      NaN      NaN       NaN  36017.0  ...                  NaN   
11422      NaN      NaN       NaN  36017.0  ...                  NaN   
11433      NaN      NaN       NaN  36023.0  ...                  NaN   
11434      NaN      NaN       NaN  36023.0  ...                  NaN   
11446      NaN      NaN       NaN  36030.0  ...                  NaN   

       start_y_y  end_x_y  end_y_y  length_y  inverter_line N or S or M  \
11421        NaN      NaN      NaN       

In [35]:
script_df = groups_df[['GRP','No. of Strings', 'Duct Definition', 'Number of Ducts', 'closest_y', 'X', 'inverter_x', 'inverter_y','Line', 'inverter_line', 'STR_CUM_TO_INV', 'Y-N-STR', 'Y-S-STR', 'N or S or M']].copy()
script_df.rename(columns={'closest_y': 'Y'}, inplace=True)

distance3 = 1.6968 # Vertical distance between the tracker's pile and the inverter.
distance4 = 0.5325 # Distance between the middle point of the inverter and its edge on the X axis.
distance5 = 0.2 # Distance between the middle point of the inverter and its edge on the Y axis.
distance6 = 1 # Distance to make an intermediary point for stretching purposes.
distance7 = 1.5 # Distance to make an intermediary point for stretching purposes.

def intermediary_coordinates(df):
    df['intermediary_x_1'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['X'] + distance3,
            df['X'] - distance3
        )
    )

    df['intermediary_x_2'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance6,
            df['inverter_x'] + distance6
        )
    )

    df['intermediary_x_3'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance7,
            df['inverter_x'] + distance7
        )
    )

    df['inverter_x'] = np.where(
        df['inverter_line'] == df['Line'],
        df['inverter_x'],
        np.where(
            df['inverter_x'] > df['X'],
            df['inverter_x'] - distance4,
            df['inverter_x'] + distance4
        )
    )

    df['inverter_y'] = np.where(
        df['inverter_line'] != df['Line'],
        df['inverter_y'],
        np.where(
            df['N or S or M'] == 'S',
            df['inverter_y'] + distance5,
            df['inverter_y'] - distance5
        )
    )

'''

CABOS

'''

intermediary_coordinates(cables_df)

def calculate_cable_length(row, color):
    try:
        x = row['X']
        y_n_str = row['N exit']
        y_s_str = row['S exit']
        closest_y = row['closest_y']
        int_x_1 = row['intermediary_x_1']
        inv_y = row['inverter_y']
        inv_x = row['inverter_x']

        if color == 'red':
            seg1 = math.sqrt((x - x)**2 + (y_n_str - closest_y)**2)
        else:
            seg1 = math.sqrt((x - x)**2 + (y_s_str - closest_y)**2)
        seg2 = math.sqrt((x - int_x_1)**2 + (closest_y - inv_y)**2)
        seg3 = math.sqrt((int_x_1 - inv_x)**2 + (inv_y - inv_y)**2)

        return seg1 + seg2 + seg3
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise

cables_df['Red Cable Length'] = cables_df.apply(lambda row: calculate_cable_length(row, 'red'), axis=1)
cables_df['Black Cable Length'] = cables_df.apply(lambda row: calculate_cable_length(row, 'black'), axis=1)

total_red_length = cables_df['Red Cable Length'].sum()
total_black_length = cables_df['Black Cable Length'].sum()

total_cable_length = total_red_length + total_black_length

cables_df['Script Red Cables'] = cables_df.apply(
    lambda row: f"_.pline {row['X']},{row['N exit']} {row['X']},{row['closest_y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)
cables_df['Script Black Cables'] = cables_df.apply(
    lambda row: f"_.pline {row['X']},{row['S exit']} {row['X']},{row['closest_y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\paiva\Script Cabos Tracker.scr", 'w') as f:
    f.write('PLINEWID 0\n')
    f.write("LAYER S EMF_cabo_solar_positivo\n\n")
    f.write('\n'.join(cables_df["Script Red Cables"].astype(str)) + '\n')
    f.write("LAYER S EMF_cabo_solar_negativo\n\n")
    f.write('\n'.join(cables_df["Script Black Cables"].astype(str)) + '\n')

'''

VALAS

'''

intermediary_coordinates(script_df)

script_df['Script Valas'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} ",
    axis=1
)
print(script_df['Script Valas'])

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\paiva\Script Valas Tracker.scr", 'w') as f:
    f.write('PLINEWID 0.5\n')
    f.write("LAYER S EMF_bt_vala\n\n")
    f.write('\n'.join(script_df["Script Valas"].astype(str)) + '\n')


'''

DUTOS

'''

def calculate_duct_length(row, no_of_strings):
    try:
        x = row['X']
        y = row['Y']
        int_x = row['intermediary_x_1']
        inv_y = row['inverter_y']
        inv_x = row['inverter_x']
        number_of_ducts = row['Duct Definition'].count(no_of_strings)

        seg1 = math.sqrt((x - int_x)**2 + (y - inv_y)**2)
        seg2 = math.sqrt((int_x - inv_x)**2 + (inv_y - inv_y)**2)

        return (seg1 + seg2) * number_of_ducts
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise


script_df['Dutos 1 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '1'), axis=1)
script_df['Dutos 2 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '2'), axis=1)
script_df['Dutos 3 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '3'), axis=1)
script_df['Dutos 4 Length'] = script_df.apply(lambda row: calculate_duct_length(row, '4'), axis=1)

total_1str_duct_length = script_df['Dutos 1 Length'].sum()
total_2str_duct_length = script_df['Dutos 2 Length'].sum()
total_3str_duct_length = script_df['Dutos 3 Length'].sum()
total_4str_duct_length = script_df['Dutos 4 Length'].sum()

total_duct_length = total_1str_duct_length + total_2str_duct_length + total_3str_duct_length + total_4str_duct_length

script_df['Script Dutos 1'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('1'),
    axis=1
)
script_df['Script Dutos 2'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('2'),
    axis=1
)
script_df['Script Dutos 3'] = script_df.apply(
    lambda row: '\n'.join([
        f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} "
        for _ in range(str(row['Duct Definition']).count('3'))
    ]),
    axis=1
)
script_df['Script Dutos 4'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} {row['intermediary_x_3']},{row['inverter_y']} {row['intermediary_x_2']},{row['inverter_y']} {row['inverter_x']},{row['inverter_y']} " * str(row['Duct Definition']).count('4'),
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\paiva\Script Dutos Tracker.scr", 'w') as f:
    f.write('PLINEWID 0.03\n')
    # Layer 1
    layer1_commands = script_df["Script Dutos 1"].astype(str)
    layer1_commands = layer1_commands[layer1_commands != '']  # Remove empty strings
    if len(layer1_commands) > 0:  # Only write layer if there are commands
        f.write("LAYER S EMF_bt_eletroduto_1S\n\n")
        f.write('\n'.join(layer1_commands) + '\n')

    # Layer 2
    layer2_commands = script_df["Script Dutos 2"].astype(str)
    layer2_commands = layer2_commands[layer2_commands != '']
    if len(layer2_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_2S\n\n")
        f.write('\n'.join(layer2_commands) + '\n')

    # Layer 3
    layer3_commands = script_df["Script Dutos 3"].astype(str)
    layer3_commands = layer3_commands[layer3_commands != '']
    if len(layer3_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_3S\n\n")
        f.write('\n'.join(layer3_commands) + '\n')

    # Layer 4
    layer4_commands = script_df["Script Dutos 4"].astype(str)
    layer4_commands = layer4_commands[layer4_commands != '']
    if len(layer4_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_4S\n\n")
        f.write('\n'.join(layer4_commands) + '\n')

4346    _.pline 36016.90525000314,129710.0716877195 36...
4351    _.pline 36023.20525000313,129710.0716877195 36...
4356    _.pline 36029.50525000313,129710.0716877195 36...
4361    _.pline 36035.80525000313,129710.0716877195 36...
4367    _.pline 36042.10525000313,129710.0716877195 36...
                              ...                        
575     _.pline 34387.48615000009,125998.2946993383 34...
590     _.pline 34393.7861500001,125998.2946993383 343...
604     _.pline 34400.0861500001,125998.2946993383 343...
618     _.pline 34406.38615000008,125998.2946993383 34...
633     _.pline 34412.68615000008,125998.2946993383 34...
Name: Script Valas, Length: 4719, dtype: str


In [36]:
'''

LAYERED VALAS

'''

pitch = 6.3

script_df['intermediary_x_4'] = np.where(
    script_df['inverter_line'] == script_df['Line'],
    script_df['inverter_x'],
    np.where(
        script_df['inverter_x'] > script_df['X'],
        script_df['intermediary_x_1'] + pitch,
        script_df['intermediary_x_1'] - pitch
    )
)

script_df['final_x'] = np.where(
    script_df['inverter_line'] == script_df['Line'],
    script_df['inverter_x'],
    np.where(
        script_df['inverter_x'] > script_df['X'],
        script_df.apply(lambda row: min(row['intermediary_x_4'], row['inverter_x']), axis=1),
        script_df.apply(lambda row: max(row['intermediary_x_4'], row['inverter_x']), axis=1)
    )
)

def calculate_trench_length(row):
    try:
        x = row['X']
        y = row['Y']
        int_x = row['intermediary_x_1']
        inv_y = row['inverter_y']
        final_x = row['final_x']

        seg1 = math.sqrt((x - int_x)**2 + (y - inv_y)**2)
        seg2 = math.sqrt((int_x - final_x)**2 + (inv_y - inv_y)**2)

        return seg1 + seg2
    except KeyError as e:
        print(f"KeyError: {e} - Available keys: {row.index.tolist()}")
        raise


script_df['Valas Length'] = script_df.apply(calculate_trench_length, axis=1)

total_trench_length = script_df['Valas Length'].sum()

script_df['Script Valas Layered 1'] = script_df.apply(
    lambda row: f"_.pline {row['X']},{row['Y']} {row['intermediary_x_1']},{row['inverter_y']} ",
    axis=1
)

script_df['Script Valas Layered 2'] = script_df.apply(
    lambda row: f"_.pline {row['intermediary_x_1']},{row['inverter_y']} {row['final_x']},{row['inverter_y']} ",
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\paiva\Script Valas Layered Tracker.scr", 'w') as f:
    f.write('PLINEWID 0.5\n')
    for i in range(1, max_number_of_strings+1):
        mask1 = script_df['No. of Strings'] == i
        mask2 = script_df['STR_CUM_TO_INV'] == i

        if mask1.any() or mask2.any():
            f.write(f"LAYER S EMF_vala_CC_{i}C\n\n")

            if mask1.any():
                f.write('\n'.join(script_df[mask1]["Script Valas Layered 1"].astype(str)) + '\n')

            if mask2.any():
                f.write('\n'.join(script_df[mask2]["Script Valas Layered 2"].astype(str)) + '\n')


lengths_df = pd.DataFrame.from_dict(data={
    'DC Red Cable Length': [total_red_length],
    'DC Black Cable Length': [total_black_length],
    'DC Total Cable Length': [total_cable_length],
    '1str Duct Length': [total_1str_duct_length],
    '2str Duct Length': [total_2str_duct_length],
    '3str Duct Length': [total_3str_duct_length],
    '4str Duct Length': [total_4str_duct_length],
    'Total Duct Length': [total_duct_length],
    'Total Trench Length': [total_trench_length]
}, orient="index", columns=['Length (m)'])
lengths_df.to_excel(r"C:\Users\Usuario\Desktop\Lengths.xlsx")
script_df.to_excel(r"C:\Users\Usuario\Desktop\SCRIPTS.xlsx")
cables_df.to_excel(r"C:\Users\Usuario\Desktop\cables.xlsx")